####Paso 1 - Leer el archivo JSON usando "DataFrameReader de Spark"

In [0]:
dbutils.widgets.text("p_file_date", "2024-12-16")
v_file_date = dbutils.widgets.get("p_file_date")

In [0]:
%run "../includes/configurations"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
language_role_schema = StructType(fields=[
  StructField("roleId", IntegerType(), True),
  StructField("languageRole", StringType(), True)
])

In [0]:
language_role_df = spark.read \
  .schema(language_role_schema) \
  .option("multiline", True) \
  .json(f"{bronze_folder_path}/{v_file_date}/language_role.json")

In [0]:
display(language_role_df)

####Paso 2 - Renombrar las columnas y añadir nuevas columnas

In [0]:
from pyspark.sql.functions import current_timestamp, lit

In [0]:
language_role_final_df = language_role_df \
    .withColumnsRenamed({"roleId": "rol_id",
                        "languageRole": "language_Role"}) \
    .withColumn("ingestion_date", current_timestamp()) \
    .withColumn("enviroment", lit("Produccion")) \
    .withColumn("file_date", lit(v_file_date))

In [0]:
display(language_role_final_df)

####Paso 3 - Escribir la salida en formato "Parquet"

In [0]:
#language_role_final_df.write.mode("overwrite").parquet("abfss://silver@moviehistorybymg.dfs.core.windows.net/languages_role")
language_role_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.languages_role")

In [0]:
%sql
SELECT *
FROM movie_silver.languages_role

In [0]:
#%fs
#ls abfss://silver@moviehistorybymg.dfs.core.windows.net/languages_role

In [0]:
#display(spark.read.parquet("abfss://silver@moviehistorybymg.dfs.core.windows.net/languages_role"))